# 1. Importing Libraries and Environment Setup
Importing all necessary libraries for data processing, augmentations, neural network construction (PyTorch, Segmentation Models PyTorch), and metrics calculation.

In [ ]:
!pip install segmentation_models_pytorch -q

In [ ]:

import pandas as pd
import segmentation_models_pytorch as smp
import cv2
import numpy as np
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader,Dataset
import os
import kagglehub
import torch.optim as optim
import torch
from google.colab import drive
from tqdm import tqdm
import gc

In [ ]:
df_folds=pd.read_csv("/content/train_folds.csv")
df_folds

,ImageId,class_1,class_2,class_3,class_4,defects_count,fold
0,0002cc93b.jpg,1,0,0,0,1,3
1,00031f466.jpg,0,0,0,0,0,4
2,000418bfc.jpg,0,0,0,0,0,3
3,000789191.jpg,0,0,0,0,0,1
4,0007a71bf.jpg,0,0,1,0,1,2
...,...,...,...,...,...,...,...
12563,fff0295e1.jpg,0,0,0,0,0,4
12564,fff02e9c5.jpg,0,0,1,0,1,1
12565,fffe98443.jpg,0,0,1,0,1,4
12566,ffff4eaa8.jpg,0,0,1,0,1,4


# 2. Data Loading and Dataset Initialization
Downloading the dataset via Kaggle Hub, setting up paths, and creating the custom `SteelDataset` class to handle images and masks.

In [ ]:
# 1. Installing libraries
!pip install -q segmentation-models-pytorch albumentations kagglehub

# 2. Importing core libraries
import os
import kagglehub

# 3. Kaggle API authentication
os.environ["KAGGLE_USERNAME"] = "narekgabrielyan"
os.environ["KAGGLE_KEY"] = "66235ac7057c609edc6803926b672559"

# 4. Downloading data
path = kagglehub.competition_download('severstal-steel-defect-detection')

# 5. Fixing paths
train_img_dir = os.path.join(path, 'train_images')
train_csv_path = os.path.join(path, 'train.csv')

# 6. Loading and checking the folds file
labels_df = pd.read_csv(train_csv_path)

print("Data is ready:")
print(f"Images folder: {train_img_dir}")
print(f"df_folds shape: {df_folds.shape}")
print(f"labels_df shape: {labels_df.shape}")

Data is ready:
Images folder: /root/.cache/kagglehub/competitions/severstal-steel-defect-detection/train_images
df_folds shape: (12568, 7)
labels_df shape: (7095, 3)


In [ ]:
train_df=df_folds[df_folds.fold!=0]
val_df=df_folds[df_folds.fold==0]

In [ ]:
# 1. RLE Decoder function (gets a 256x1600 binary mask from text code)
def rle_decode(mask_rle, shape=(256, 1600)):
    if pd.isna(mask_rle) or mask_rle == '':
        return np.zeros(shape, dtype=np.uint8)

    s = mask_rle.split()
    starts = np.asarray(s[0::2], dtype=int) - 1
    lengths = np.asarray(s[1::2], dtype=int)
    ends = starts + lengths

    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    return img.reshape(shape, order='F')

# 2. PyTorch Dataset Class
class SteelDataset(Dataset):
    def __init__(self, df, labels_df, img_dir, transforms=None):
        self.df = df.reset_index(drop=True)
        self.labels_df = labels_df
        self.img_dir = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['ImageId']
        img_path = os.path.join(self.img_dir, img_id)

        # Loading the image in RGB format
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Creating a 4-channel mask (256, 1600, 4)
        mask = np.zeros((256, 1600, 4), dtype=np.float32)
        img_labels = self.labels_df[self.labels_df['ImageId'] == img_id]

        for _, row in img_labels.iterrows():
            class_idx = int(row['ClassId']) - 1
            mask[:, :, class_idx] = rle_decode(row['EncodedPixels'])

        # Augmentations
        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # (H, W, 4) -> (4, H, W) for PyTorch
        mask = mask.permute(2, 0, 1)

        return image, mask

# 3. Augmentation Pipeline (Albumentations)
Defining transformations for the training and validation subsets (resizing, normalization, conversion to PyTorch tensors).

In [ ]:
# 1. Transformations: WITHOUT Resize
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
    ToTensorV2(),
])

# 2. DataLoaders: batch_size = 4
train_loader = DataLoader(
    SteelDataset(
        train_df, labels_df, train_img_dir, transforms=train_transform
    ),
    batch_size=4,  # optimal for Colab GPU with 256x1600
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    SteelDataset(val_df, labels_df, train_img_dir, transforms=val_transform),
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# 3. Checking dimensions
sample_imgs, sample_masks = next(iter(train_loader))
print(f"Batch Images shape: {sample_imgs.shape}")  # (4, 3, 256, 1600)
print(f"Batch Masks shape:  {sample_masks.shape}")  # (4, 4, 256, 1600)

Batch Images shape: torch.Size([4, 3, 256, 1600])
Batch Masks shape:  torch.Size([4, 4, 256, 1600])


# 4. Model, Loss Function, and Optimizer Initialization
Setting up the neural network architecture (Unet), loss function, score computation.

In [ ]:
class CombinedLoss(nn.Module):

  def __init__(self):
    super().__init__()
    self.bce = nn.BCEWithLogitsLoss()
    self.dice = smp.losses.DiceLoss(mode="multilabel")

  def forward(self, logits, targets):
    return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)

In [ ]:
def compute_dice_score(logits, targets, threshold=0.5, eps=1e-7):
  probs = torch.sigmoid(logits)
  preds = (probs > threshold).float()

  intersection = (preds * targets).sum(dim=(2, 3))
  total_area = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))

  dice = (2.0 * intersection + eps) / (total_area + eps)
  return dice.mean().item()

# 5. Training and Validation Loop
The core training loop: processing batches, computing loss, validating on each epoch, and saving the best model weights.

In [ ]:
def train_model(model, train_loader, val_loader, model_save_name, lr=3e-4, epochs=10):
    criterion = CombinedLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_dice = 0.0

    for epoch in range(epochs):
        # 1. Training phase
        model.train()
        train_loss = 0.0

        for images, masks in train_loader:
            images = images.cuda()
            masks = masks.cuda()

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        scheduler.step()
        avg_train_loss = train_loss / len(train_loader)

        # 2. Validation phase
        model.eval()
        val_loss = 0.0
        val_dice = 0.0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.cuda()
                masks = masks.cuda()

                outputs = model(images)
                loss = criterion(outputs, masks)

                val_loss += loss.item()
                val_dice += compute_dice_score(outputs, masks)

        avg_val_loss = val_loss / len(val_loader)
        avg_val_dice = val_dice / len(val_loader)

        # 3. Logging & Saving
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Dice: {avg_val_dice:.4f}", end="")

        if avg_val_dice > best_dice:
            best_dice = avg_val_dice
            torch.save(model.state_dict(), model_save_name)
            print(" -> [✓ Best Model Saved!]")
        else:
            print()

    print(f"\nBest Val Dice = {best_dice:.4f}")




In [ ]:
# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Create a folder in Google Drive to store the models
SAVE_DIR = "/content/drive/MyDrive/Severstal_Models"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Weights have been saved in: {SAVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Weights have been saved in: /content/drive/MyDrive/Severstal_Models


In [ ]:
baseline_model = smp.Unet(
    encoder_name="resnet34",  # Powerful and optimal encoder for segmentation
    encoder_weights= None,  # Uses already learned features
    in_channels=3,
    classes=4,
    activation=None,  # Raw logits
)

baseline_model = baseline_model.cuda()

In [ ]:
# Pre-trained: More powerful ResNet34 encoder with ImageNet weights
pretrained_model = smp.Unet(
    encoder_name="resnet34",  # Powerful and optimal encoder for segmentation
    encoder_weights="imagenet",  # Uses already learned features
    in_channels=3,
    classes=4,
    activation=None,  # Raw logits
)

pretrained_model = pretrained_model.cuda()

In [ ]:
# import torch
# # Վերցնում ենք Օղակ 2-ի ստուգման batch-ը
# sample_imgs, sample_masks = next(iter(train_loader))
# sample_imgs = sample_imgs.cuda()

# # Անցկացնում ենք մոդելով
# with torch.no_grad():
#   output_test = pretrained_model(sample_imgs)

# print("✓ Օղակ 3-ը հաջողությամբ պատրաստ է:")
# print(f"  Input Tensor Shape:  {sample_imgs.shape}")  # torch.Size([4, 3, 256, 1600])
# print(
#     f"  Output Logits Shape: {output_test.shape}"
# )  # torch.Size([4, 4, 256, 1600])

✓ Օղակ 3-ը հաջողությամբ պատրաստ է:
  Input Tensor Shape:  torch.Size([4, 3, 256, 1600])
  Output Logits Shape: torch.Size([4, 4, 256, 1600])


In [ ]:
print("=== 1. Baseline U-Net (Scratch) ===")
train_model(
    model=baseline_model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_save_name="baseline_unet.pth",
    epochs=10,
    lr=3e-4
)

=== 1. Baseline U-Net (Scratch) ===
Epoch [01/10] | Train Loss: 0.1673 | Val Loss: 0.1432 | Val Dice: 0.8422 -> [✓ Best Model Saved!]


In [ ]:
print("\n=== 2. Pre-trained ResNet34 U-Net ===")
train_model(
    model=pretrained_model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_save_name="pretrained_resnet34_unet.pth",
    epochs=15,
    lr=3e-4
)


=== 2. Pre-trained ResNet34 U-Net ===
Epoch [01/15] | Train Loss: 0.1592 | Val Loss: 0.1383 | Val Dice: 0.6291 -> [✓ Best Model Saved!]
Epoch [02/15] | Train Loss: 0.1232 | Val Loss: 0.1454 | Val Dice: 0.5227
Epoch [03/15] | Train Loss: 0.1113 | Val Loss: 0.1235 | Val Dice: 0.6566 -> [✓ Best Model Saved!]


# 6. Inference and Result Visualization
Loading trained weights, running inference on test or validation images, getting scores.

In [ ]:
baseline_model_path = "/content/drive/MyDrive/Severstal_Models/baseline_resnet34_unet_last.pth"

In [ ]:
pretrained_model_path = "/content/drive/MyDrive/Severstal_Models/pretrained_resnet34_unet_last.pth"

In [ ]:
baseline_check = torch.load(baseline_model_path)

In [ ]:
pretrained_check = torch.load(pretrained_model_path)

In [ ]:
pretrained_check.keys()

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'best_dice', 'val_dice'])

In [ ]:
baseline_model.load_state_dict(baseline_check["model_state_dict"])


<All keys matched successfully>

In [ ]:
baseline_model.eval()

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [ ]:
pretrained_model.load_state_dict(pretrained_check["model_state_dict"])


<All keys matched successfully>

In [ ]:
pretrained_model.eval()

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 4. Optional: If you want to compute the Dice score for validation
def compute_dice_per_batch(preds, targets, smooth=1e-6):
    intersection = (preds * targets).sum(axis=(2, 3))
    union = preds.sum(axis=(2, 3)) + targets.sum(axis=(2, 3))
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return dice.mean()

In [ ]:
def run_inference_and_evaluate(model, dataloader, device, threshold=0.5,smooth=1e-6):
    """
    Runs inference on a validation or test dataloader and computes predictions/targets.
    """
    model.eval()
    total_dice = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Running Evaluation"):
            images, masks = batch[0].to(device), batch[1].to(device)

            # Forward pass: get model outputs (logits)
            outputs = model(images)

            # Apply sigmoid and binarize
            probs = torch.sigmoid(outputs)
            preds = (probs > threshold).float()

            # Compute Dice score for the current batch
            # Sum over spatial dimensions (height and width)
            intersection = (preds * masks).sum(dim=(2, 3))
            union = preds.sum(dim=(2, 3)) + masks.sum(dim=(2, 3))
            dice = (2.0 * intersection + smooth) / (union + smooth)

            total_dice += dice.mean().item()
            num_batches += 1

            # Clear cache to free up VRAM/RAM
            del images, masks, outputs, probs, preds
            torch.cuda.empty_cache()
    mean_dice = total_dice / num_batches
    return mean_dice


In [ ]:
baseline_model.to(device)

val_dice = run_inference_and_evaluate(baseline_model, val_loader, device)
print(f"Validation Dice Score: {val_dice:.4f}")

# Final cleanup
gc.collect()

Running Evaluation: 100%|██████████| 629/629 [01:29<00:00,  7.02it/s]


Validation Dice Score: 0.7090


16

In [ ]:
pretrained_model.to(device)

val_dice = run_inference_and_evaluate(pretrained_model, val_loader, device)
print(f"Validation Dice Score: {val_dice:.4f}")

# Final cleanup
gc.collect()

Running Evaluation: 100%|██████████| 629/629 [01:32<00:00,  6.76it/s]


Validation Dice Score: 0.8068


0

# Conclusion: Impact of Pretraining
The evaluation results clearly demonstrate that random initialization performs significantly worse on this dataset. The pretrained model achieves a validation Dice score of **0.8068**, outperforming the baseline model (**0.7090**) by nearly 10%. This highlights how crucial pretrained weights are for extracting low-level features (such as edges and textures) in complex defect segmentation tasks, whereas training completely from scratch struggles to capture these patterns effectively.